# Create Matched Filter Templates for Utah Arrays
This notebook connects to a Blackrock session, applies channel mapping, highpass filters 
the data, runs the spike sorter (default: spykingcircus2) and extracts peak templates.
Because notebooks are interactive, we define the parameters here.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np

import spikeinterface as si
import spikeinterface.preprocessing as spre
import spikeinterface.extractors as se
import spikeinterface.sorters as ss
import spikeinterface_gui as sig

In [2]:
# Add parent dir to path so we can import RCP_analysis
REPO_ROOT = Path(os.getcwd()).resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import RCP_analysis as rcp
from RCP_analysis.python.functions.config_loading import *

In [3]:
# Parameters
SESSION_NAME = 'NRR_RW022_001'
SORTER = 'spykingcircus2'
DURATION_SECONDS = 60.0  # Set to 0 to run on the entire file

In [4]:
# Load Session
sess_path = BR_ROOT / SESSION_NAME.replace('.ns6', '').replace('.nsx6', '')
if not sess_path.with_suffix('.ns6').exists() and not sess_path.exists():
    raise FileNotFoundError(f"Session file not found at {sess_path}.ns6")

print(f"Loading Blackrock session {SESSION_NAME}...")
rec_ns6 = se.read_blackrock(sess_path, stream_name='nsx6', all_annotations=True)

Loading Blackrock session NRR_RW022_001...


In [5]:
# Apply Channel Mapping
clean_sess_name = SESSION_NAME.replace('.ns6', '').replace('.nsx6', '')
try:
    br_idx = int(clean_sess_name.split('_')[-1])
except ValueError:
    print("Could not parse numeric BR index from session name. Defaulting to 001")
    br_idx = 1
    
XLS = rcp.ua_excel_path(REPO_ROOT, PARAMS.probes)
UA_MAP = rcp.load_UA_mapping_from_excel(XLS) if XLS else None

if UA_MAP is None:
    raise RuntimeError("UA mapping required.")
    
rec_ns6, idx_rows, ua_elec, ua_nsp, ua_region, ua_region_names, ua_port = rcp.apply_ua_mapping_with_regions(
    rec_ns6, UA_MAP, br_idx, METADATA_CSV
)

[MAP] UA port from metadata: 'B'
[MAP] Applying NSP offset for Port B: local 1..128 -> NSP 129..256
[MAP] renamed 128/128 rows with UA mapping ('UAe###_NSP###') using port 'B'.


In [6]:
# Filter and validate channels
print("Applying highpass filter...")
rec_hp = spre.highpass_filter(rec_ns6, freq_min=float(PARAMS.highpass_hz))

valid_mask = ua_elec > 0
ch_ids = rec_hp.get_channel_ids()
valid_ch_ids = [ch_ids[i] for i, valid in enumerate(valid_mask) if valid]

print(f"Keeping {len(valid_ch_ids)} valid mapped channels...")
rec_valid = rec_hp.select_channels(valid_ch_ids)

port_a_ch = valid_ch_ids if ua_port == 'A' else []
port_b_ch = valid_ch_ids if ua_port == 'B' else []

if DURATION_SECONDS > 0 and rec_valid.get_total_duration() > DURATION_SECONDS:
    print(f"Slicing down to first {DURATION_SECONDS} seconds to speed up sorting...")
    rec_valid = rec_valid.frame_slice(0, int(DURATION_SECONDS * rec_valid.get_sampling_frequency()))

Applying highpass filter...
Keeping 128 valid mapped channels...
Slicing down to first 60.0 seconds to speed up sorting...


In [9]:
# Process Port A (Run Sorter & Analyzer)
results_dir = UA_CKPT_ROOT / f"mf_templates_{clean_sess_name}"
results_dir.mkdir(exist_ok=True, parents=True)
conf_dir = REPO_ROOT / 'config'
conf_dir.mkdir(exist_ok=True)

# Select Port
port_name = "PortA"
port_ch = port_a_ch
analyzer_A = None

if len(port_ch) > 0:
    print(f"--- Processing {port_name} ({len(port_ch)} channels) ---")
    rec_port = rec_valid.select_channels(port_ch)
    
    # CMR
    rec_port = spre.common_reference(rec_port, reference="global", operator="median")

    # Set generic 2D channel locations (400um pitch typical for UA)
    n_ch = rec_port.get_num_channels()
    locs = np.zeros((n_ch, 2))
    side = int(np.ceil(np.sqrt(n_ch)))
    for i in range(n_ch):
        locs[i, 0] = (i % side) * 400.0
        locs[i, 1] = (i // side) * 400.0
    rec_port.set_channel_locations(locs)
    rec_port.set_property("group", np.zeros(n_ch, dtype=int))

    sorting_out = results_dir / f"sorting_{port_name}"
    sorting = None
    if sorting_out.exists():
        try:
            sorting = si.load_extractor(sorting_out / 'sorter_output')
        except:
            try:
                sorting = si.load_extractor(sorting_out)
            except:
                import shutil
                shutil.rmtree(sorting_out, ignore_errors=True)
                
    if sorting is None:
        print(f"Running {SORTER} on {port_name}...")
        sorting = ss.run_sorter(SORTER, rec_port, folder=str(sorting_out), remove_existing_folder=True, verbose=True)
    
    analyzer_out = results_dir / f"analyzer_{port_name}"
    if not analyzer_out.exists():
        analyzer_A = si.create_sorting_analyzer(sorting, rec_port, format="binary_folder", folder=str(analyzer_out), overwrite=True)
        analyzer_A.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
        analyzer_A.compute("waveforms", ms_before=1.5, ms_after=2.0)
        analyzer_A.compute("templates")
        try:
            analyzer_A.compute("quality_metrics")
        except:
            pass
    else:
        analyzer_A = si.load_sorting_analyzer(str(analyzer_out))
        
    print(f"Found {len(analyzer_A.unit_ids)} units in {port_name}")
else:
    print(f"No channels mapped to {port_name}")

No channels mapped to PortA


### Explore Port A in GUI
Run this cell to open the SpikeInterface GUI. **Make sure to close the GUI window when you are done to proceed.**

In [ ]:
if analyzer_A is not None:
    app = sig.run_mainwindow(analyzer_A)

In [ ]:
# Extract Port A Template
# Set the best_id based on your observation from the GUI.
best_id_A = "PUT_UNIT_ID_HERE"  # Change this to an integer or string based on GUI

if analyzer_A is not None and best_id_A != "PUT_UNIT_ID_HERE":
    ext_templates = analyzer_A.get_extension("templates").get_data()
    unit_idx = list(analyzer_A.unit_ids).index(best_id_A)
    template_2d = ext_templates[unit_idx]
    
    ext_dict = si.get_template_extremum_channel(analyzer_A, peak_sign='neg')
    ext_ch = ext_dict[best_id_A]
    ext_ch_idx = rec_port.id_to_index(ext_ch)
    
    template_1d = template_2d[:, ext_ch_idx]
    norm_factor = np.abs(np.min(template_1d))
    template_1d_norm = template_1d / norm_factor
    
    out_path = conf_dir / f"median_extremum_templates_norm_UA_PortA.npy"
    np.save(out_path, template_1d_norm)
    print(f"Saved Port A template to {out_path}")

In [10]:
# Process Port B
port_name = "PortB"
port_ch = port_b_ch
analyzer_B = None

if len(port_ch) > 0:
    print(f"--- Processing {port_name} ({len(port_ch)} channels) ---")
    rec_port_b = rec_valid.select_channels(port_ch)

    n_ch = rec_port_b.get_num_channels()
    locs = np.zeros((n_ch, 2))
    side = int(np.ceil(np.sqrt(n_ch)))
    for i in range(n_ch):
        locs[i, 0] = (i % side) * 400.0
        locs[i, 1] = (i // side) * 400.0
    rec_port_b.set_channel_locations(locs)
    rec_port_b.set_property("group", np.zeros(n_ch, dtype=int))

    sorting_out_b = results_dir / f"sorting_{port_name}"
    sorting_b = None
    if sorting_out_b.exists():
        try:
            sorting_b = si.load_extractor(sorting_out_b / 'sorter_output')
        except:
            try:
                sorting_b = si.load_extractor(sorting_out_b)
            except:
                import shutil
                shutil.rmtree(sorting_out_b, ignore_errors=True)
                
    if sorting_b is None:
        print(f"Running {SORTER} on {port_name}...")
        sorting_b = ss.run_sorter(SORTER, rec_port_b, folder=str(sorting_out_b), remove_existing_folder=True, verbose=True)
    
    analyzer_out_b = results_dir / f"analyzer_{port_name}"
    if not analyzer_out_b.exists():
        analyzer_B = si.create_sorting_analyzer(sorting_b, rec_port_b, format="binary_folder", folder=str(analyzer_out_b), overwrite=True)
        analyzer_B.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
        analyzer_B.compute("waveforms", ms_before=1.5, ms_after=2.0)
        analyzer_B.compute("templates")
        try:
            analyzer_B.compute("quality_metrics")
        except:
            pass
    else:
        analyzer_B = si.load_sorting_analyzer(str(analyzer_out_b))
        
    print(f"Found {len(analyzer_B.unit_ids)} units in {port_name}")
else:
    print(f"No channels mapped to {port_name}")

--- Processing PortB (128 channels) ---
Running spykingcircus2 on PortB...
Preprocessing the recording (bandpass filtering + CMR + whitening)
Geometry of the probe does not allow 1D drift correction


noise_level (no parallelization):   0%|          | 0/20 [00:00<?, ?it/s]

write_memory_recording (no parallelization):   0%|          | 0/60 [00:00<?, ?it/s]

get protoype waveforms (no parallelization):   0%|          | 0/60 [00:00<?, ?it/s]

detect peaks (matched_filtering) 
engine=process - n_jobs=1 - samples_per_chunk=30,000 - chunk_memory=14.65 MiB - total_memory=14.65 MiB - chunk_duration=1.00s


...

Removed 0 empty templates
Removed 0 unaligned templates
Removed 0 templates with too low SNR
Removed 46 templates with too high mean sd / noise ratio
Kept 88 raw clusters


e:\conda_install\envs\pipeline\lib\site-packages\spikeinterface\postprocessing\template_similarity.py:345: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  overlapping_ids = overlapping_j_list[i]
e:\conda_install\envs\pipeline\lib\site-packages\spikeinterface\core\baserecordingsnippets.py:259: UserWarning: There is no Probe attached to this recording. Creating a dummy one with contact positions
  warn("There is no Probe attached to this recording. Creating a dummy one with contact positions")
e:\conda_install\envs\pipeline\lib\site-packages\spikeinterface\core\baserecordingsnippets.py:259: UserWarning: There is no Probe attached to this recording. Creating a dummy one with contact positions
  warn("There is no Probe attached to this recording. Creating a dummy one with contact positions")


Kept 88 non-duplicated clusters
Removed 0 empty templates
Removed 0 unaligned templates
Removed 0 templates with too low SNR
Removed 0 templates with too high mean sd / noise ratio
remove_small_cluster: kept  81 removed 7 (min_spike_count 6)
Kept 81 clean clusters
find spikes (circus-omp) 
engine=process - n_jobs=1 - samples_per_chunk=30,000 - chunk_memory=14.65 MiB - total_memory=14.65 MiB - chunk_duration=1.00s


find spikes (circus-omp) (no parallelization):   0%|          | 0/60 [00:00<?, ?it/s]

Found 139886 spikes
Kept 81 units after final merging
spykingcircus2 run time 278.03s
Found 85 units in PortB


### Explore Port B in GUI

In [11]:
if analyzer_B is not None:
    app = sig.run_mainwindow(analyzer_B)

In [ ]:
# Extract Port B Template
best_id_B = "PUT_UNIT_ID_HERE"

if analyzer_B is not None and best_id_B != "PUT_UNIT_ID_HERE":
    ext_templates = analyzer_B.get_extension("templates").get_data()
    unit_idx = list(analyzer_B.unit_ids).index(best_id_B)
    template_2d = ext_templates[unit_idx]
    
    ext_dict = si.get_template_extremum_channel(analyzer_B, peak_sign='neg')
    ext_ch = ext_dict[best_id_B]
    ext_ch_idx = rec_port_b.id_to_index(ext_ch)
    
    template_1d = template_2d[:, ext_ch_idx]
    norm_factor = np.abs(np.min(template_1d))
    template_1d_norm = template_1d / norm_factor
    
    out_path = conf_dir / f"median_extremum_templates_norm_UA_PortB.npy"
    np.save(out_path, template_1d_norm)
    print(f"Saved Port B template to {out_path}")

In [12]:
# %%
# Extract Port B Template using Median of Hand-Curated Units
import numpy as np
import spikeinterface as si
import matplotlib.pyplot as plt

good_units_B = [1, 2, 11, 22, 29, 30, 37, 38, 39, 40, 42, 43, 47, 51, 52, 55, 61, 
                62, 70, 71, 74, 79, 81, 86, 92, 93, 103, 112, 113, 117, 118, 125]

if analyzer_B is not None:
    ext_templates = analyzer_B.get_extension("templates").get_data() # (n_units, n_samples, n_channels)
    ext_dict = si.get_template_extremum_channel(analyzer_B, peak_sign='neg')
    
    good_1d_templates = []
    
    for u_id in good_units_B:
        # Some sorters use string IDs instead of ints, handle it gracefully
        if u_id not in analyzer_B.unit_ids:
            u_id = str(u_id)
            
        if u_id in analyzer_B.unit_ids:
            unit_idx = list(analyzer_B.unit_ids).index(u_id)
            ch_id = ext_dict[u_id]
            ch_idx = rec_port_b.id_to_index(ch_id)
            
            # Slice out the 1D waveform on the peak channel
            template_1d = ext_templates[unit_idx, :, ch_idx]
            good_1d_templates.append(template_1d)
        else:
            print(f"Warning: Unit {u_id} not found in the analyzer.")

    if good_1d_templates:
        # Stack all the curated 1D templates
        good_1d_templates = np.stack(good_1d_templates)
        
        # Normalize each template so its max negative peak is precisely -1.0
        norm_factors = np.abs(np.min(good_1d_templates, axis=1, keepdims=True))
        norm_factors[norm_factors == 0] = 1.0 # Prevent division by zero
        
        good_templates_norm = good_1d_templates / norm_factors
        
        # Compute the pure median archetype
        median_curated_template = np.median(good_templates_norm, axis=0)
        
        out_path = conf_dir / "median_extremum_templates_norm_UA_PortB.npy"
        np.save(out_path, median_curated_template)
        
        print(f"-> SUCCESS! Computed pure median from {len(good_1d_templates)} hand-curated units.")
        print(f"-> Saved template to {out_path}")
        
        # Plot it to visually verify
        plt.figure(figsize=(6, 3))
        for t in good_templates_norm:
            plt.plot(t, color='grey', alpha=0.3, lw=0.5) # Plot all the good individuals faintly
            
        plt.plot(median_curated_template, color='blue', lw=2) # Overlay the bold median line
        plt.title("Median Template of Curated Port B Units")
        plt.xlabel("Samples")
        plt.ylabel("Normalized Amplitude")
        plt.show()
    else:
        print("Error: None of the curated units were found.")


-> SUCCESS! Computed pure median from 32 hand-curated units.
-> Saved template to E:\NHP_Cerebellum_Project_2025_RCP_Analysis_Git\RCP_analysis\config\median_extremum_templates_norm_UA_PortB.npy


C:\Users\culle\AppData\Local\Temp\ipykernel_5888\1665836293.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
